# 02 - Model development walkthrough

Forecasting with conformal intervals, calibrated risk model + SHAP, anomaly detection, simulation sanity checks.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd, numpy as np
from common import settings
panel = pd.read_parquet(settings.PROCESSED_DIR / 'monthly_panel.parquet')
train = pd.read_parquet(settings.PROCESSED_DIR / 'training_dataset.parquet')

In [ ]:
from ml.forecasting.dataset import build_forecast_dataset
ds = build_forecast_dataset(panel, target_col='expense', horizons=(1,3,6,12))
print('rows', len(ds.X), 'features', ds.X.shape[1])
ds.X.head()

In [ ]:
from ml.forecasting.expense_forecast import train_forecast_model
bundle = train_forecast_model(panel, target='expense', horizons=(1,3,6,12), algorithm='xgboost')
{h: m.metrics for h, m in bundle.horizons.items()}

In [ ]:
from ml.risk.risk_model import train_risk_model
rm = train_risk_model(train, algorithm='xgboost')
rm.metrics['roc_auc'], rm.metrics['brier'], list(rm.metrics['top_features'].items())[:5]

In [ ]:
from ml.explainability.shap_explain import explain
from ml.explainability.counterfactual import counterfactuals
row = train.sort_values('dna_financial_risk', ascending=False).iloc[0]
X = pd.DataFrame([row])[rm.features].astype(float)
ex = explain(rm, X)
[(c['label'], c['points']) for c in ex['contributions'][:6]]

In [ ]:
cf = counterfactuals(rm, row)
cf['current_score'], [(o['label'], o['new_score'], o['delta']) for o in cf['options'][:4]]

In [ ]:
from ml.anomaly_detection.anomaly import train_anomaly_model
tx = pd.read_parquet(settings.PROCESSED_DIR / 'transactions.parquet')
am = train_anomaly_model(tx, sample_size=400_000)
am.metrics

In [ ]:
from simulation.monte_carlo import UserState, SimConfig, ScenarioParams, run_simulation
st = UserState.from_features(row)
base = run_simulation(st, ScenarioParams(), SimConfig(n_paths=5000, horizon_months=36))
car  = run_simulation(st, ScenarioParams(lump_sum_expense=1_000_000, new_debt_principal=3_000_000,
                                         new_debt_rate=0.20, new_debt_term_months=60),
                      SimConfig(n_paths=5000, horizon_months=36))
pd.DataFrame([base.summary, car.summary], index=['baseline','car'])[
    ['goal_probability','stress_probability','expected_final_net_worth','p5_final_net_worth']]